In [2]:
import pandas as pd
import re

In [3]:
df = pd.read_csv('../sql/amazon.csv')
df.columns

Index(['product_id', 'product_name', 'category', 'discounted_price',
       'actual_price', 'discount_percentage', 'rating', 'rating_count',
       'about_product', 'user_id', 'user_name', 'review_id', 'review_title',
       'review_content', 'img_link', 'product_link'],
      dtype='str')

In [4]:
def clean_func(value):
    str_val = str(value).strip()  # 去掉前后空格
     # 去掉逗号和其他非数字字符（除了小数点）
    str_val = re.sub(r'[^\d.]', '', str_val)
    return str_val


In [5]:

# 清洗价格字段
df['discounted_price'] = df['discounted_price'].apply(clean_func)
df['actual_price'] = df['actual_price'].apply(clean_func)
df['rating_count'] = df['rating_count'].apply(clean_func)

In [ ]:
def clean_text(text):
    # Remove punctuation and convert to lowercase
    # return ''.join(e for e in text if e.isalnum() or e.isspace()).lower()
    if pd.isna(text):
        return ''  # 处理缺失值，返回空字符串
    
    text = str(text)

    # 移除控制字符（保留换行、制表符）
    # 移除不可打印字符（即ASCII码小于32的字符，除了换行（9）和制表符（10）还有回车（13））
    cleaned = ''
    for char in text:
        if char.isprintable() or char in ['\n', '\t', '\r']:
            cleaned += char # 即为cleaned = cleaned + char
    return cleaned
    # 这是设置黑名单的方式，保留可打印字符和特定的控制字符（换行、制表符、回车），其他都移除掉。
    # 但仍然会有遗漏。

# 另一种清洗
def clean_text_v2(text):
    cleaned = ''
    for char in text:
        code = ord(char)
        # 明确允许的字符范围
        if (code == 9 or code == 10 or code == 13 or      # 制表符、换行、回车
            (code >= 32 and code <= 0xD7FF) or            # 基本多文种平面
            (code >= 0xE000 and code <= 0x10FFFF)):      # 其他Unicode平面
            cleaned += char
        else:
            cleaned += ' '  # 替换为空格

    # 移除BOM字符
    cleaned = cleaned.replace('\ufeff', '').replace('\u200b', '')
    
    # 标准化空白字符
    cleaned = ' '.join(cleaned.split())
    
    return cleaned


def clean_for_sql(text):
    """清理文本，确保SQL Server兼容"""
    if pd.isna(text):
        return ''
    
    text = str(text)
    
    # 移除BOM和零宽度字符
    text = text.replace('\ufeff', '').replace('\u200b', '')

    #把|替换成空格，避免SQL中的分隔符问题
    text = text.replace('|', ' ')
    
    # 替换SQL不支持的字符
    # 常见的Unicode字符替换
    replacements = {
        '\u2018': "'",  # 左单引号
        '\u2019': "'",  # 右单引号
        '\u201c': '"',  # 左双引号
        '\u201d': '"',  # 右双引号
        '\u2013': '-',  # 短横线
        '\u2014': '--', # 长横线
        '\u2026': '...',# 省略号
    }
    
    for old, new in replacements.items():
        text = text.replace(old, new)
    
    # 移除非ASCII字符（保留基本字符）
    # 保留：字母、数字、常见标点、中文
    cleaned = []
    for char in text:
        code = ord(char)
        if (32 <= code <= 126 or       # 基本ASCII
            0x4E00 <= code <= 0x9FFF or # 中文
            char in ['\n', '\t', '\r', ' ']):
            cleaned.append(char)
        else:
            cleaned.append(' ')  # 替换为空格
    
    return ''.join(cleaned)



In [7]:
df['about_product'].apply(clean_text)
df['about_product'].apply(clean_text_v2)
df['about_product'].apply(clean_for_sql)

0       High Compatibility : Compatible With iPhone 12...
1       Compatible with all Type C enabled devices, be...
2         Fast Charger& Data Sync -With built-in safet...
3       The boAt Deuce USB 300 2 in 1 cable is compati...
4       [CHARGE & SYNC FUNCTION]- This cable comes wit...
                              ...                        
1460    SUPREME QUALITY 90 GRAM 3 LAYER THIK PP SPUN F...
1461                         230 Volts, 400 watts, 1 Year
1462    International design and styling Two heat sett...
1463    Fan sweep area: 230 MM ; Noise level: (40 - 45...
1464    Brand-Borosil, Specification   " 23V ~ 5Hz;1 W...
Name: about_product, Length: 1465, dtype: str

In [8]:

# 与原文件同目录下生成清洗后的CSV文件
df.to_csv('../sql/amazon_cleaned.csv', index=False, encoding='utf-8-sig')

In [ ]:
# 查看数据的review_id列的最大长度
max_length = df['review_id'].apply(lambda x: len(str(x))).max()
print(f"review_id列的最大长度: {max_length}")
# 设置Varchar（255）足够了

review_id列的最大长度: 119


In [10]:
len('yield')

5